# NB-04: T-8 ペースカテゴリ予測（H/M/S）

**ターゲット**: レースのペースが H(ハイ) / M(ミドル) / S(スロー) のどれかを予測する。  
**粒度**: レース単位（1レースに1ラベル）  
**モデル**: LightGBM 3クラス分類  
**主評価指標**: F1-macro

### ラベル導出
`pace` JSON の `first_half_3f - second_half_3f` で分類:
- H: diff > +1.0 秒（前半が速い）
- S: diff < -1.0 秒（後半が速い/スロー逃げ）
- M: その間


In [ ]:
import sys
sys.path.insert(0, "/home/jovyan/work/keiba-vpn")
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, lightgbm as lgb
import matplotlib.pyplot as plt

from src.pipeline.models.notebook_utils import (
    load_master, load_raw_results, encode_cats, feature_set,
    train_lgb, oof_predict, eval_classification, save_oof,
    DEFAULT_PARAMS_MULTICLASS, SURFACE_CATS, derive_pace_category,
)
print("imports OK")


## 1. データ読み込みとラベル生成

In [ ]:
df = load_master()

# pace をレース単位で取得
rr = load_raw_results(["race_id","horse_number","pace"])
rr = rr.drop_duplicates(subset=["race_id","horse_number"])

df = df.merge(rr[["race_id","horse_number","pace"]], on=["race_id","horse_number"], how="left")
df["pace_cat"] = df["pace"].map(derive_pace_category)

print("ペースカテゴリ分布:")
print(df["pace_cat"].value_counts())
print(f"ラベルなし率: {df['pace_cat'].isna().mean():.1%}")

# レース単位でラベルを集約（1レース=1ラベル）
race_pace = (
    df[df["pace_cat"].notna()]
    .groupby("race_id")["pace_cat"]
    .first()
    .reset_index()
)
print(f"\nレース数 (ラベルあり): {len(race_pace):,}")
df["surface_cat"] = df["surface_cat"].astype(str)  # Categorical → str


## 2. レース単位集計特徴量の作成

In [ ]:
# ペース予測の特徴量はレース単位の集計値
# 各レース内の平均オッズ、騎手勝率の分散、フィールドサイズ等
race_feats = (
    df.groupby("race_id").agg(
        distance    = ("distance", "first"),
        field_size  = ("field_size", "first"),
        venue_code  = ("venue_code", "first"),
        surface     = ("surface", "first"),
        direction   = ("direction", "first"),
        grade       = ("grade", "first"),
        track_condition = ("track_condition", "first"),
        weather     = ("weather", "first"),
        split       = ("split", "first"),
        surface_cat = ("surface_cat", "first"),
        jk_avg_win_rate   = ("jk_prior_all_win_rate", "mean"),
        jk_avg_pass_first = ("jk_prior_all_avg_pass_first", "mean"),
        jk_std_win_rate   = ("jk_prior_all_win_rate", "std"),
        jk_max_win_rate   = ("jk_prior_all_win_rate", "max"),
        tr_avg_win_rate   = ("tr_prior_all_win_rate", "mean"),
        speed_avg_race    = ("speed_avg", "mean"),
        speed_max_race    = ("speed_max", "max"),
    ).reset_index()
)

# ペースラベルをマージ
race_feats = race_feats.merge(race_pace, on="race_id", how="inner")
print(f"レース集計テーブル: {race_feats.shape}")
print(race_feats["pace_cat"].value_counts())


## 3. 馬場別モデル学習

In [ ]:
label_map = {"H": 0, "M": 1, "S": 2}
rev_map   = {v: k for k, v in label_map.items()}
params    = {**DEFAULT_PARAMS_MULTICLASS, "num_class": 3}

FEAT_T8 = [c for c in [
    "distance", "field_size", "venue_code",
    "surface", "direction", "grade", "track_condition", "weather",
    "jk_avg_win_rate", "jk_avg_pass_first", "jk_std_win_rate", "jk_max_win_rate",
    "tr_avg_win_rate", "speed_avg_race", "speed_max_race",
] if c in race_feats.columns]
print("特徴量:", FEAT_T8)

CAT_USE = [c for c in ["surface","direction","grade","track_condition","weather"] if c in race_feats.columns]
race_feats = encode_cats(race_feats, CAT_USE)

models_t8: dict = {}
oof_all_t8 = pd.DataFrame()

for sc in SURFACE_CATS:
    df_sc = race_feats[race_feats["surface_cat"] == sc].copy()
    df_tr = df_sc[df_sc["split"] == "train"]
    df_vl = df_sc[df_sc["split"] == "valid"]
    print(f"\n=== {sc} ===  train={len(df_tr):,}  valid={len(df_vl):,}")
    if len(df_tr) < 50:
        print("  スキップ")
        continue

    model = train_lgb(df_tr, df_vl, FEAT_T8, "pace_cat", params,
                      cat_features=CAT_USE, label_encoder=label_map)
    models_t8[sc] = model

    X_vl = df_vl[FEAT_T8]
    pred_class = model.predict(X_vl).argmax(axis=1)
    y_vl = df_vl["pace_cat"].map(label_map)
    mask = y_vl.notna()
    metrics = eval_classification(y_vl[mask].values, pred_class[mask.values], task="multiclass")
    print(f"  Valid: {metrics}")

    proba_tr = model.predict(df_tr[FEAT_T8])
    oof_df = df_tr[["race_id","surface_cat","pace_cat","split"]].copy()
    for ci, label in enumerate(["H","M","S"]):
        oof_df[f"t8_prob_{label}"] = proba_tr[:, ci]
    oof_all_t8 = pd.concat([oof_all_t8, oof_df], ignore_index=True)

print("\nモデル学習完了:", list(models_t8.keys()))


## 4. テスト評価 & OOF 保存

In [ ]:
for sc, model in models_t8.items():
    df_te = race_feats[(race_feats["surface_cat"] == sc) & (race_feats["split"] == "test")]
    if df_te.empty:
        continue
    pred_class = model.predict(df_te[FEAT_T8]).argmax(axis=1)
    y_te = df_te["pace_cat"].map(label_map).values
    metrics = eval_classification(y_te, pred_class, task="multiclass")
    print(f"[{sc}] Test: {metrics}")

# OOF 保存 (horse 単位に展開してマージできるよう race_id キーで保存)
if not oof_all_t8.empty:
    oof_all_t8.to_parquet("/home/jovyan/work/keiba-vpn/data/local/modeling/oof/t8_oof.parquet", index=False)
    print(f"\nT-8 OOF saved: shape={oof_all_t8.shape}")

# 特徴量重要度
best_sc = max(models_t8, key=lambda s: models_t8[s].num_trees()) if models_t8 else None
if best_sc:
    imp = pd.Series(models_t8[best_sc].feature_importance(importance_type="gain"), index=FEAT_T8)
    imp.sort_values(ascending=False).head(10).plot.barh(title=f"T-8 Importance [{best_sc}]")
    plt.tight_layout(); plt.show()
